# 03 快速分析（读 PostgreSQL）
不需要连 TWS。数据来自 `bars` 表，五只 ETF 2019 年至今日线。

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from ibkr_quant.db import read_closes, engine
from ibkr_quant.analysis.returns import summary, simple_returns
import sqlalchemy as sa

ETFS = ['SPY', 'QQQ', 'IWM', 'TLT', 'GLD']
pd.read_sql(sa.text('select symbol, count(*) as rows, min(ts)::date as first, max(ts)::date as last from bars group by 1 order by 1'), engine)

In [ ]:
closes = read_closes(ETFS).dropna()
closes.index = closes.index.tz_localize(None)
closes.tail()

In [ ]:
summary(closes).round(4)

In [ ]:
nav = (1 + simple_returns(closes)).cumprod()
nav.plot(figsize=(11,5), grid=True, title='Cumulative return since 2019 (rebased to 1)');

In [ ]:
simple_returns(closes).corr().round(3)

In [ ]:
# 按年收益
yearly = closes.resample('YE').last().pct_change().dropna()
yearly.index = yearly.index.year
(yearly*100).round(1)